# H.J. Andrews Experimental Forest: Forest Recovery from Timber Harvest 1985–2025
### Using USFS Landscape Change Monitoring System (LCMS) in Google Earth Engine

---

**Research Question:** How have harvested forest areas in the H.J. Andrews Experimental Forest recovered over the LCMS period (1985–2024), and do recovery trajectories differ between areas with distinct harvest histories?

**Dataset:** [LCMS v2025-11](https://developers.google.com/earth-engine/datasets/catalog/projects_gtac-data-publish_assets_LCMS_Product_Version_2025-11) — USFS/GTAC annual land cover, land use, and change maps at 30 m, 1985–2025 (CONUS + SE Alaska).

LCMS produces three annual thematic products:
| Band | What it shows |
|------|--------------|
| `Land_Cover` | What is on the ground (Trees, Shrubs, Grass, Barren, Water, etc.) |
| `Land_Use` | How the land is used (Forest, Agriculture, Developed, Rangeland, etc.) |
| `Change` | What changed and how (Tree Removal, Successional Growth, Wildfire, Stable, etc.) |

**Why the H.J. Andrews?**  
The [H.J. Andrews Experimental Forest](https://andrewsforest.oregonstate.edu/) is a Long-Term Ecological Research (LTER) site in the western Cascade Range of Oregon, operated by Oregon State University and the USDA Forest Service. Its well-documented history of experimental and commercial timber harvest — ranging from 1950s clearcutting to 2000s variable-retention cuts — makes it an ideal site for studying post-harvest forest recovery as captured by LCMS.

LCMS covers 1985–2025, which means:
- Areas harvested **before 1985** appear at the *start* of the record already in early-to-mid recovery
- Areas harvested **during 1985–2025** show their full post-harvest trajectory captured in LCMS
- Undisturbed old-growth stands provide a stable reference baseline throughout

**Prerequisites**
- A Google Earth Engine (GEE) account — [sign up here](https://earthengine.google.com/signup/)
- A GEE Cloud project ID — [create one here](https://console.cloud.google.com/projectcreate)
- Python ≥ 3.9 with `earthengine-api` and `geeViz` installed:
  ```bash
  pip install earthengine-api geeViz
  ```

> 💡 **Workshop note:** This notebook is designed to run sequentially top-to-bottom. All cells are self-documenting. Run `Kernel → Restart & Run All` for a clean start.

## 1 · Setup — Imports and Authentication

In [ ]:
import os, ee
from IPython.display import display, HTML

# ── Authentication ────────────────────────────────────────────────────────────
# Run this once per machine/account to store credentials locally.
# After that, comment it out and just call ee.Initialize() below.
#ee.Authenticate()

# ── Initialization ────────────────────────────────────────────────────────────
# ee.Initialize MUST come before geeViz imports so the eeAuth proxy starts
# with the correct project. Importing geeViz first causes it to spin up the
# proxy with whatever cached credentials it finds (potentially from another project).
# Replace 'your-project-id' with your GEE Cloud project ID.
EE_PROJECT = 'rcr-gee'
ee.Initialize(project=EE_PROJECT)

# ── Force in-process HTTP server ──────────────────────────────────────────────
# Without this, geeViz spawns a *detached* eeAuth subprocess from whatever
# Python is on the system PATH. That subprocess serves its own copy of the
# geeViz package directory. The notebook's venv writes run_geeViz.js to the
# venv copy, but the detached process serves the system-Python copy — so the
# browser always fetches stale layer state (e.g. SRTM from a previous run).
# "auto" keeps the HTTP server in-process (same venv, same file paths).
os.environ['GEEVIZ_EEAUTH_MODE'] = 'auto'

# ── geeViz imports ────────────────────────────────────────────────────────────
import geeViz.getImagesLib as gil
import geeViz.geeView
import geeViz.getSummaryAreasLib as sal
import geeViz.esriLib as el
from geeViz.outputLib import charts as cl

# -- Set up geeViz Map object --------------------------------------------------
Map = gil.Map # assign geeViz Map object to variable for convenience
Map.port = 8080
Map.project = EE_PROJECT
Map.clearMap()

# ── HJA Boundary — USFS ArcGIS Feature Service ───────────────────────────────
# Official H.J. Andrews Experimental Forest perimeter (layer 9 of the service).
HJA_REST_URL = 'https://services1.arcgis.com/CD5mKowwN6nIaqd8/arcgis/rest/services'

HJA_BOUNDARY_URL   = f'{HJA_REST_URL}/HJ_Andrews_Base_GIS_Layers__2018/FeatureServer/9'

# ── Local GeoJSON copies (downloaded to data/ — no live service call needed) ──
import pathlib, json
_repo_root = pathlib.Path(__file__).parent.parent if '__file__' in dir() else pathlib.Path().resolve().parent
HJA_WATERSHEDS_GEOJSON = _repo_root / 'data' / 'hja_watersheds.geojson'

# Load watersheds as an EE FeatureCollection from the local file
with open(HJA_WATERSHEDS_GEOJSON) as f:
    _ws_gj = json.load(f)
hja_watersheds_fc = ee.FeatureCollection(_ws_gj['features'])

print(f'Watersheds loaded: {hja_watersheds_fc.size().getInfo()} features')

# -- Test Earth Engine connection ------------------------------------------------
test = ee.Image(1).getInfo()
print(test, '\n Earth Engine initialized successfully.')


{'type': 'Image', 'bands': [{'id': 'constant', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 1, 'max': 1}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}]} 
 Earth Engine initialized successfully.


## 2 · Study Area — H.J. Andrews Experimental Forest

The [H.J. Andrews Experimental Forest](https://andrewsforest.oregonstate.edu/) encompasses approximately 15,800 acres (6,400 ha) in the western Cascade Range of Oregon, within the Willamette National Forest roughly 50 miles east of Eugene. It is drained entirely by Lookout Creek, a tributary of the McKenzie River.

The forest has a uniquely well-documented land management history:

| Period | Management Activity | LCMS Visibility |
|--------|--------------------|----|
| 1950s–1970s | Large-scale clearcutting of old-growth Douglas-fir; experimental watershed treatments | Recovery already underway at 1985 LCMS start |
| 1980s–early 1990s | Declining commercial harvest; shift to variable-retention approaches | Harvest events visible in LCMS Change band |
| 1994–present | Northwest Forest Plan → most of forest designated Late-Successional Reserve; near halt to cutting | Predominantly "Stable" and "Successional Growth" |
| 2002–2010 | Limited salvage logging and fuels treatments | Tree Removal detectable for specific units |

Because each harvest area represents a different starting point for recovery, comparing them side by side in LCMS reveals both the *pace* and *completeness* of Pacific Northwest forest regeneration.

> 📍 **Polygon note:** The harvest unit boundaries used in this notebook are approximate, based on publicly documented watershed descriptions. For analysis using verified boundaries, download GIS data from the [HJ Andrews data portal](https://andrewsforest.oregonstate.edu/data/gis).

In [2]:
# -- Study date range: ------------------------------------------------------
# LCMS data extend from 1984 through 2025. The 2025 collection is the most recent.
START_YEAR = 1984
END_YEAR   = 2025

# ── Primary study area: H.J. Andrews Experimental Forest bounding box ─────────
# The actual forest is ~64 km²; this bbox includes surrounding Willamette NF context.
study_area = ee.Geometry.BBox(-122.32, 44.18, -122.09, 44.28)

# ── Option B: Willamette National Forest (the administrative parent forest) ────
# Uncomment to use the broader Willamette NF boundary instead.
# willamette_nf = sal.getUSFSForests(forest_name='Willamette')
# study_area    = willamette_nf.geometry().convexHull(maxError=500)


# -- Print study area info ------------------------------------------------------
print("Study date range:", START_YEAR, "to", END_YEAR)
print('Study area type:', study_area.getInfo()['type'])
area_km2 = study_area.area(maxError=500).divide(1e6).getInfo()
print(f'Bounding box area : {area_km2:,.0f} km²  (H.J. Andrews ≈ 64 km²; bbox includes surrounding forest)')


Study date range: 1984 to 2025
Study area type: Polygon
Bounding box area : 204 km²  (H.J. Andrews ≈ 64 km²; bbox includes surrounding forest)


## 3 · Load LCMS Data

LCMS v2024-10 covers 1985–2024 (40 years). Each image in the collection represents one calendar year. We filter to the H.J. Andrews bounding box and inspect what's available.

For the western Oregon Cascades, the LCMS `Change` band classes most relevant to this analysis are:

| Change Class | Ecological Meaning |
|---|---|
| **Tree Removal** | Timber harvest or clearing — the primary disturbance of interest |
| **Vegetation Successional Growth** | Forest recovery — canopy closure and stand development after disturbance |
| **Wildfire** | Fire disturbance (lower frequency in the wet western Cascades) |
| **Stable** | No detected change — mature/undisturbed forest holds this class for decades |

> 🔍 The cell below prints the full class-value mapping so you can see the exact numeric IDs used in your LCMS version. These values are read directly from the image metadata rather than hardcoded.

In [3]:
LCMS_ASSET_FOR_PROPERTIES = 'USFS/GTAC/LCMS/v2024-10'
LCMS_ASSET = 'projects/gtac-data-publish/assets/LCMS/Product_Version/2025-11'

# Filter to study area
lcms = ee.ImageCollection(LCMS_ASSET).filterBounds(study_area)

# Basic metadata
n_images = lcms.size().getInfo()
years    = lcms.aggregate_array('year').distinct().sort().getInfo()
bands    = lcms.first().bandNames().getInfo()

print(f'Images in collection : {n_images}')
print(f'Years                : {years[0]}–{years[-1]}')
print(f'Bands                : {bands}')

# ── Print Change class mapping ──────────
# Change class mapping is stored in the older LCMS asset, which is used for metadata only.
sample_img    = ee.ImageCollection(LCMS_ASSET_FOR_PROPERTIES).first()
change_names  = sample_img.get('Change_class_names').getInfo()
change_values = sample_img.get('Change_class_values').getInfo()
change_palettes = sample_img.get('Change_class_palette').getInfo()

# create dictionary mapping change values to names for later use
change_viz_dict = {val: name for val, name in zip(change_values, change_names)}

# ── Apply 2024-10 symbology to the 2025 collection ────────────────────────────
from geeViz.examples.lcmsLevelLookup import getLevelNRemap, all_lookup_2024_10


# Build a single props dict for all three bands 
# Uses deepest level available in the lookup for each band
viz_props = {}
for band in ['Land_Cover', 'Land_Use', 'Change']:
    level = max(len(k.split('-')) for k in all_lookup_2024_10[band].keys())
    viz_props.update(getLevelNRemap(level, band, all_lookup_2024_10)['viz_dict'])

# Stamp the props onto every image in the 2025 collection
lcms = lcms.map(lambda img: img.set(viz_props))

Images in collection : 41
Years                : 1985–2025
Bands                : ['Change', 'Change_Raw_Probability_Slow_Loss', 'Change_Raw_Probability_Fast_Loss', 'Change_Raw_Probability_Gain', 'Land_Cover', 'Land_Cover_Raw_Probability_Trees', 'Land_Cover_Raw_Probability_Tall-Shrubs-and-Trees-Mix', 'Land_Cover_Raw_Probability_Shrubs-and-Trees-Mix', 'Land_Cover_Raw_Probability_Grass-Forb-Herb-and-Trees-Mix', 'Land_Cover_Raw_Probability_Barren-and-Trees-Mix', 'Land_Cover_Raw_Probability_Tall-Shrubs', 'Land_Cover_Raw_Probability_Shrubs', 'Land_Cover_Raw_Probability_Grass-Forb-Herb-and-Shrubs-Mix', 'Land_Cover_Raw_Probability_Barren-and-Shrubs-Mix', 'Land_Cover_Raw_Probability_Grass-Forb-Herb', 'Land_Cover_Raw_Probability_Barren-and-Grass-Forb-Herb-Mix', 'Land_Cover_Raw_Probability_Barren-or-Impervious', 'Land_Cover_Raw_Probability_Snow-or-Ice', 'Land_Cover_Raw_Probability_Water', 'Land_Use', 'Land_Use_Raw_Probability_Agriculture', 'Land_Use_Raw_Probability_Developed', 'Land_Use_Raw_Prob

c:\Users\LILA\Documents\GitHub\lcms-outreach\.venv\Lib\site-packages\ee\deprecation.py:215: DeprecationWarning: 

Attention required for USFS/GTAC/LCMS/v2024-10! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by projects/gtac-data-publish/assets/LCMS/Product_Version/2025-11

Learn more: https://developers.google.com/earth-engine/datasets/catalog/USFS_GTAC_LCMS_v2024-10

  warnings.warn(warning, category=DeprecationWarning)


## 4 · Interactive Map — Current Forest State and Harvest History

The map below shows three layers for the H.J. Andrews:

1. **Land Cover 2024** — what is on the ground today; most of the mature forest shows "Trees"
2. **Change 2024** — the most recent annual change classification
3. **Ever Harvested 1985–2024** — pixels flagged as "Tree Removal" in *any* year of the 40-year LCMS record

Use the **Ever Harvested** layer to orient yourself to the spatial pattern of harvest before defining the comparison polygons in Section 5.

> 🗺️ Toggle layers on/off with the layer panel on the right. Click any pixel to query its class value. Draw a polygon and click "Chart Selected Area" to analyze custom sub-regions.

In [4]:
Map.clearMap()

lcms_most_recent = lcms.filter(ee.Filter.eq('year', END_YEAR))

# Land Cover 2025
Map.addLayer(
    lcms_most_recent.select('Land_Cover'),
    {'autoViz': True, 'canAreaChart': True},
    f'Land Cover {END_YEAR}',
    True,
)

# Change 2025
lcms_most_common_change = ee.Image(lcms.select('Change').mode()).clip(study_area)
Map.addLayer(
    lcms_most_common_change,
    {'autoViz': True, 'canAreaChart': True},
    f'Change {END_YEAR}',
    False,
)

# ── Dynamically find the Tree Removal class value from image metadata ──────────
name_to_val = dict(zip(change_names, change_values))
TREE_REMOVAL_VAL = name_to_val.get(
    'Tree Removal',
    name_to_val.get('Non-Fire Mechanical', 5),
)
print(f'Tree Removal class value in this LCMS version: {TREE_REMOVAL_VAL}')

# ── "Ever Harvested" — any pixel with Tree Removal detected in any LCMS year ──
ever_harvested = (
    lcms.select('Change')
    .map(lambda img: img.eq(TREE_REMOVAL_VAL))
    .max()
    .selfMask()
    .rename('ever_harvested')
)
ever_harvested = ever_harvested.set({
    'ever_harvested_class_values':  [1],
    'ever_harvested_class_names':   ['Tree Removal detected (any year 1985–2025)'],
    'ever_harvested_class_palette': ['c47b1e'],
})
Map.addLayer(ever_harvested, {'autoViz': True}, 'Ever Harvested 1985–2025', True)

# HJA official boundary perimeter from USFS ArcGIS Feature Service
el.addEsriFeatureService(
    HJA_BOUNDARY_URL,
    viz_params={'strokeColor': 'ffff00', 'strokeWidth': 2.5, 'fillColor': '00000000'},
    name='HJA Boundary',
)

# Study area bounding box (approximate; used for LCMS filtering context)
Map.addLayer(
    ee.Feature(study_area, {}),
    {'layerType': 'geeVector', 'strokeColor': 'ffffff', 'strokeWidth': 1,
     'fillColor': '00000000'},
    'Study Area BBox',
    False,
)

Map.setCenter(-122.20, 44.23, 11)
Map.view()


Adding layer: Land Cover 2025
Adding layer: Change 2025
Tree Removal class value in this LCMS version: 9
Adding layer: Ever Harvested 1985–2025
Adding Esri Feature Service: HJA Boundary (4 features)
Adding layer: HJA Boundary
Adding layer: Study Area BBox
Starting webmap


[geeViz.eeAuth] EE initialized via proxy: http://127.0.0.1:8889/ee-api (tenant_header=X-geeViz-Creds)


geeViz server at http://localhost:8080/geeView/
Using eeCreds proxy at http://127.0.0.1:8889/ee-api (creds=ee-persistent)
geeView URL: http://localhost:8080/geeView/?v=1784244756324


## 5 · Harvest Comparison Polygons

We define four representative harvest areas that bracket different stages of the 1985–2024 recovery arc. Each polygon roughly corresponds to a documented experimental watershed or harvest unit within H.J. Andrews.

| Polygon | Approx. Harvest | Age at LCMS Start (1985) | Expected LCMS Signal |
|---------|----------------|--------------------------|---------------------|
| **WS1 area** | Clearcut 1962–1966 | ~20–23 yrs | Young closed-canopy trees already establishing; Trees % high and rising |
| **WS6 area** | Clearcut ~1974 | ~11 yrs | Dense shrub / early seral; Trees % should rise steadily through record |
| **Central harvest** | ~1988–1993 | Harvested at onset | Tree Removal spike in Change band; Shrub peak → Trees recovery captured |
| **Upper basin harvest** | ~2003–2008 | Post-1985 | Full trajectory in record: Tree Removal → Grass → Shrub → Trees |

> ⚠️ These polygons are **approximate** and intended for comparative education. Verify against the [HJ Andrews GIS portal](https://data-osugisci.opendata.arcgis.com/) before using for formal analysis. Adjust the `BBox` coordinates to match official harvest unit shapefiles.

In [ ]:
# ── Define harvest comparison areas ───────────────────────────────────────────
# Keys = display labels; values = approximate bounding boxes.
# Adjust coordinates to match verified harvest unit shapefiles.
harvest_areas = {
    'WS1 area (clearcut 1962–1966)':       ee.Geometry.BBox(-122.289, 44.186, -122.261, 44.202),
    'WS6 area (clearcut ~1974)':            ee.Geometry.BBox(-122.275, 44.198, -122.248, 44.213),
    'Central harvest (~1988–1993)':         ee.Geometry.BBox(-122.248, 44.210, -122.213, 44.228),
    'Upper basin harvest (~2003–2008)':     ee.Geometry.BBox(-122.225, 44.228, -122.180, 44.258),
}

# Watershed polygons from local GeoJSON (loaded in setup cell)
# Exclude the top-level Lookout Creek catchment so only sub-watersheds display
ws_display = hja_watersheds_fc.filter(ee.Filter.neq('Wshed_Name', 'Lookout'))
Map.addLayer(
    ws_display,
    {'layerType': 'geeVectorImage', 'strokeColor': '38a8e8', 'strokeWidth': 2.5,
     'fillColor': '00000000'},
    'HJA Watersheds',
)

# Quick area check
print('Harvest polygon areas:')
for name, geom in harvest_areas.items():
    area = geom.area(maxError=100).divide(1e6).getInfo()
    print(f'  {name:45s}: {area:.2f} km²')

# ── Colors for each polygon ────────────────────────────────────────────────────
POLY_COLORS = {
    'WS1 area (clearcut 1962–1966)':       'e8a838',
    'WS6 area (clearcut ~1974)':            'e86038',
    'Central harvest (~1988–1993)':         '38a8e8',
    'Upper basin harvest (~2003–2008)':     '38e878',
}

# ── Add polygon outlines to the existing map from Section 4 ───────────────────
for name, geom in harvest_areas.items():
    c = POLY_COLORS[name]
    Map.addLayer(
        ee.Feature(geom, {}),
        {'layerType': 'geeVector', 'strokeColor': c, 'strokeWidth': 3,
         'fillColor': c + '35'},
        name,
    )

# HJA official boundary perimeter from USFS ArcGIS Feature Service
el.addEsriFeatureService(
    HJA_BOUNDARY_URL,
    viz_params={'strokeColor': 'ffff00', 'strokeWidth': 2.5, 'fillColor': '00000000'},
    name='HJA Boundary',
)

Map.setCenter(-122.22, 44.22, 11)
Map.view()


Adding Esri Feature Service: HJA Watersheds (9 features)
Adding layer: HJA Watersheds
Harvest polygon areas:
  WS1 area (clearcut 1962–1966)                : 3.97 km²
  WS6 area (clearcut ~1974)                    : 3.59 km²
  Central harvest (~1988–1993)                 : 5.58 km²
  Upper basin harvest (~2003–2008)             : 11.96 km²
Adding layer: WS1 area (clearcut 1962–1966)
Adding layer: WS6 area (clearcut ~1974)
Adding layer: Central harvest (~1988–1993)
Adding layer: Upper basin harvest (~2003–2008)
Adding Esri Feature Service: HJA Boundary (4 features)
Adding layer: HJA Boundary
Starting webmap
Using eeCreds proxy at http://127.0.0.1:8889/ee-api (creds=ee-persistent)
geeView URL: http://localhost:8080/geeView/?v=1784244758061


## 6 · Recovery Trajectories — Land Cover Through Time

For each harvest area we compute the **annual percentage of each Land Cover class** from 1985 to 2024. The classic post-clearcut succession in Pacific Northwest Douglas-fir forests follows a predictable pathway:

| Stage | Dominant Cover | Typical Post-Harvest Timing |
|-------|---------------|-----------------------------|
| **Pioneer** | Grass/Forb/Herb | Years 1–5 |
| **Early shrub** | Shrubs (vine maple, red alder, salal) | Years 3–15 |
| **Young forest** | Trees (Douglas-fir canopy closing) | Years 10–30 |
| **Closed canopy** | Trees + Tall Trees and Shrubs | Years 25+ |

Where a polygon sits in this sequence **in 1985** reflects how far along recovery was before the LCMS record began. Polygons harvested in the early 1960s should open with high Trees %; those harvested in the late 1980s–2000s should show the full Grass → Shrub → Trees arc within the chart.

> 🔍 Each chart below is a stacked area plot — the height of each color band shows what percentage of the polygon was in that cover class that year. Four charts, one per harvest area, let you compare recovery trajectories directly.

In [ ]:
lc_results = {}

for name, geom in harvest_areas.items():
    lcms_poly = ee.ImageCollection(LCMS_ASSET).filterBounds(geom)
    slug = (
        name[:18]
        .replace(' ', '_').replace('(', '').replace(')', '')
        .replace('~', '').replace('–', '-').strip('_')
    )
    result = cl.summarize_and_chart(
        lcms_poly,
        geometry=geom,
        band_names='Land_Cover',
        scale=30,               # 30 m — appropriate for small harvest polygons
        area_format='Percentage',
        title=f'Land Cover — {name}',
        chart_type='line',
        stacked=True,
        date_format='YYYY',
        width=950,
        height=440,
    )
    # Output results to a dictionary for later use (e.g., saving charts)
    lc_results[name] = result

    # Optionally save the chart to an html file
    # fname = f'hja_lc_{slug}.html'
    # cl.save_chart_html(result['chart'], fname)
    # print(f'Saved: {fname}')
    
    result['chart'].show()


Saved: hja_lc_WS1_area_clearcut.html


Saved: hja_lc_WS6_area_clearcut.html


Saved: hja_lc_Central_harvest.html


Saved: hja_lc_Upper_basin_harves.html


### 6a · Inspect the Recovery Data

The `summarize_and_chart()` call returns both a chart and the raw DataFrame. Below we extract the **Trees** column for all four harvest areas and display them in a single table for direct numerical comparison.

In [ ]:
import pandas as pd

# Extract the "Trees" column from each polygon's DataFrame
# Column name may be 'Trees' or include a class-value prefix like '1 — Trees'
trees_pct = {}
for name, result in lc_results.items():
    df = result['df']
    trees_col = next(
        (c for c in df.columns if 'Trees' in c and 'Tall' not in c),
        None,
    )
    if trees_col:
        trees_pct[name] = df[trees_col].round(1)
    else:
        print(f'Warning: Trees column not found for "{name}". Columns: {df.columns.tolist()}')

if trees_pct:
    trees_df = pd.DataFrame(trees_pct)
    trees_df.index.name = 'Year'
    print('Annual Trees cover (%) by harvest area:\n')
    print(trees_df.to_markdown())


## 7 · Change Agent Signatures — Tree Removal and Successional Growth

The LCMS `Change` band is the direct record of *what happened* in a pixel each year. For harvested forest we expect a distinct multi-stage signature:

1. **Tree Removal** — a spike in the year(s) of harvest activity
2. **Stable** — undisturbed years before the cut and the quiet years of early regrowth
3. **Vegetation Successional Growth** — recovery years when the forest is actively re-establishing and LCMS detects the positive trend

**What to expect by polygon:**

| Polygon | Tree Removal in LCMS? | Successional Growth? |
|---------|----------------------|----------------------|
| WS1 area (1962–1966) | Unlikely — harvest predates 1985 | Yes — growth continuing as stand matures |
| WS6 area (~1974) | Unlikely | Yes — active successional growth in early record |
| Central harvest (~1988–1993) | Yes — spike visible near onset | Yes — follows shortly after removal |
| Upper basin harvest (~2003–2008) | Yes — spike mid-record | Yes — increasingly dominant toward 2024 |

> 🔍 **Look for the lag:** How many years after Tree Removal does Successional Growth become the dominant change class? That lag interval captures the "recolonization" period when cover is low but the trajectory is already upward.

In [ ]:
change_results = {}

for name, geom in harvest_areas.items():
    lcms_poly = ee.ImageCollection(LCMS_ASSET).filterBounds(geom)
    slug = (
        name[:18]
        .replace(' ', '_').replace('(', '').replace(')', '')
        .replace('~', '').replace('–', '-').strip('_')
    )
    result = cl.summarize_and_chart(
        lcms_poly,
        geometry=geom,
        band_names='Change',
        scale=30,
        area_format='Percentage',
        title=f'Change Agents — {name}',
        chart_type='line',
        stacked=False,          # unstacked so individual classes are legible
        date_format='YYYY',
        width=950,
        height=440,
    )
    change_results[name] = result
    fname = f'hja_change_{slug}.html'
    cl.save_chart_html(result['chart'], fname)
    print(f'Saved: {fname}')
    result['chart'].show()


## 8 · Combined Comparison — Tree Cover Recovery Across All Harvest Areas

Now we overlay the **Trees cover percentage** from all four harvest areas on a single chart. This is the key comparison plot that directly answers the research question:

- **Older harvests** (WS1, WS6) should show relatively high and stable — or slowly increasing — Trees percentages throughout, because recovery was well underway before 1985.
- **Intermediate harvest** (~1988–1993) should show a dip during harvest years followed by a steady rise.
- **Most recent harvest** (~2003–2008) should show the lowest Trees values in early record years, with the steepest upward trend in recent decades as young Douglas-fir stands close the canopy.

Divergence between the trajectories tells us how much of the recovery arc LCMS can observe versus what happened off-record.

> 📈 If a polygon shows unexpectedly low Trees% for an older harvest, consider whether the bounding box captures non-harvest areas (rock outcrops, riparian corridors, roads) that dilute the tree-cover signal.

In [ ]:
import plotly.graph_objects as go

# lc_results is built in cell 12; re-run that cell if needed.

LINE_COLORS = {
    'WS1 area (clearcut 1962–1966)':       '#e8a838',
    'WS6 area (clearcut ~1974)':            '#e86038',
    'Central harvest (~1988–1993)':         '#38a8e8',
    'Upper basin harvest (~2003–2008)':     '#38e878',
}
DASH_STYLES = ['solid', 'dash', 'dot', 'dashdot']

fig = go.Figure()

for (name, result), dash in zip(lc_results.items(), DASH_STYLES):
    df = result['df']
    trees_col = next(
        (c for c in df.columns if 'Trees' in c and 'Tall' not in c),
        None,
    )
    if trees_col:
        fig.add_trace(go.Scatter(
            x=df.index,
            y=df[trees_col].values,
            name=name,
            mode='lines+markers',
            line=dict(color=LINE_COLORS[name], width=2.5, dash=dash),
            marker=dict(size=4),
        ))

fig.update_layout(
    title='H.J. Andrews — Tree Cover Recovery by Harvest Area 1985–2024',
    xaxis=dict(title='Year', tickmode='linear', dtick=5),
    yaxis=dict(title='Trees (% of polygon area)', range=[0, 100]),
    legend=dict(orientation='h', yanchor='top', y=-0.20, xanchor='left', x=0),
    width=1050,
    height=550,
    template='plotly_white',
)

fig.write_html('hja_recovery_comparison.html')
print('Comparison chart saved to hja_recovery_comparison.html')
fig.show()


## 9 · Land Use Transitions — Sankey Diagram

The Land Use Sankey shows how land area has shifted between use classes across the whole study area at key time steps. For the H.J. Andrews, watch for:

- **The dominant stable Forest block** — old-growth and recovering stands that never leave the Forest land-use class
- **Flows out of Forest** during the active timber production era (1985–1994)
- **Flows back into Forest** as harvested stands re-establish under Northwest Forest Plan protections

The transition years below align with major management inflection points:
- **1985** → LCMS baseline; active timber production still occurring
- **1995** → shortly after Northwest Forest Plan implementation (1994)
- **2010** → 15 years post-protection; second-growth stands maturing
- **2024** → present

In [ ]:
lu_sankey = cl.summarize_and_chart(
    lcms,
    geometry=study_area,
    band_names='Land_Use',
    scale=120,               # coarser scale acceptable for the full study area
    area_format='Percentage',
    title='H.J. Andrews — Land Use Transitions 1985 → 1995 → 2010 → 2024',
    sankey=True,
    transition_periods=[1985, 1995, 2010, 2024],
    min_percentage=0.5,      # hide flows < 0.5% to keep the diagram readable
    width=1000,
    height=600,
)

cl.save_chart_html(lu_sankey['chart'], 'hja_land_use_sankey.html')
print('Sankey saved to hja_land_use_sankey.html')

display(HTML(lu_sankey['chart']))


### 9a · Land Use Transition Matrices

The raw transition numbers behind the Sankey — useful for quantifying exactly how much area moved between classes in each period.

In [ ]:
if 'matrix' in lu_sankey:
    for period_key, mat in lu_sankey['matrix'].items():
        print(f'### {period_key}')
        print(mat.to_markdown())
        print()


## 10 · Interactive Time-Lapse — 40 Years of Change and Recovery

The time-lapse steps through each year's LCMS Change and Land Cover bands, letting you see *where* Tree Removal and recovery occurred spatially, not just *how much* area they affected. The harvest polygon outlines are overlaid as reference.

**What to look for:**
- The year(s) when **Tree Removal** lights up within each polygon boundary — confirms the harvest timing
- The shift from **Shrub** to **Trees** in the Land Cover time-lapse — canopy closure progressing outward from seed sources
- Adjacent undisturbed old-growth stands (which hold "Stable" throughout) providing visual contrast

> ⏱️ This may take 30–60 seconds to load — it is rendering 40 annual layers.

In [ ]:
Map.clearMap()

# Annual LCMS Change time-lapse (slider in geeViz map)
Map.addTimeLapse(
    lcms.select('Change'),
    {'autoViz': True, 'canAreaChart': True},
    'Change Agent (Annual)',
    visible=True,
)

# Annual Land Cover time-lapse
Map.addTimeLapse(
    lcms.select('Land_Cover'),
    {'autoViz': True, 'canAreaChart': True},
    'Land Cover (Annual)',
    visible=False,
)

# Harvest polygon outlines for spatial reference
for name, geom in harvest_areas.items():
    c = POLY_COLORS[name]
    Map.addLayer(
        ee.Feature(geom, {}),
        {'layerType': 'geeVector', 'strokeColor': c, 'strokeWidth': 3,
         'fillColor': '00000000'},
        name,
        False,
    )

# HJA official boundary perimeter from USFS ArcGIS Feature Service
el.addEsriFeatureService(
    HJA_BOUNDARY_URL,
    viz_params={'strokeColor': 'ffff00', 'strokeWidth': 2.5, 'fillColor': '00000000'},
    name='HJA Boundary',
)

# Study area outline
Map.addLayer(
    ee.Feature(study_area, {}),
    {'layerType': 'geeVector', 'strokeColor': 'ffffff', 'strokeWidth': 2,
     'fillColor': '00000000'},
    'H.J. Andrews Study Area',
)

Map.setCenter(-122.20, 44.23, 11)
Map.view()


## 11 · Bonus — Classify Recovery Stages Across the Study Area

Rather than looking at individual polygons, here we classify *every pixel* in the study area by how many years (out of 40) it was mapped as Trees. This "years-as-trees" metric is a proxy for successional maturity — pixels with many Trees-years are well-established forest; pixels with few Trees-years are still recovering or recently disturbed.

| Stage | Years as Trees | Interpretation |
|-------|---------------|----------------|
| **Early seral** | 0–5 | Grass/herb or shrub dominant; recent or very slow recovery |
| **Mid seral** | 6–15 | Young trees establishing but not yet dominant |
| **Late seral** | 16–29 | Dense young forest; canopy closed or closing |
| **Mature recovery** | 30–40 | Closed-canopy forest throughout nearly the entire record |

Old-growth areas (never disturbed within the LCMS period) will almost all score 30–40 and appear as **Mature recovery**. Harvest polygons will score according to their age and recovery rate.

In [ ]:
Map.clearMap()

# ── Count how many years each pixel was classified as Trees (class 1) ──────────
years_as_trees = (
    lcms.select('Land_Cover')
    .map(lambda img: img.eq(1).rename('is_trees'))
    .sum()
    .rename('years_as_trees')
)

# ── Classify into recovery stages ─────────────────────────────────────────────
# Start all pixels at 1 (Early seral), then override upward as thresholds are met.
recovery_stage = (
    ee.Image(1)
    .where(years_as_trees.gt(5),  2)   # Mid seral
    .where(years_as_trees.gt(15), 3)   # Late seral
    .where(years_as_trees.gt(29), 4)   # Mature recovery
    .rename('recovery_stage')
    .updateMask(years_as_trees.gte(0)) # keep same spatial extent
)

recovery_stage = recovery_stage.set({
    'recovery_stage_class_values':  [1, 2, 3, 4],
    'recovery_stage_class_names':   [
        'Early seral (0–5 yrs Trees)',
        'Mid seral (6–15 yrs Trees)',
        'Late seral (16–29 yrs Trees)',
        'Mature recovery (30–40 yrs Trees)',
    ],
    'recovery_stage_class_palette': ['f7dc6f', 'e67e22', '27ae60', '1a5276'],
})

# ── Add layers ────────────────────────────────────────────────────────────────
Map.addLayer(
    lcms_latest.select('Land_Cover'),
    {'autoViz': True, 'canAreaChart': True},
    'Land Cover 2025',
    False,
)
Map.addLayer(
    recovery_stage,
    {'autoViz': True, 'canAreaChart': True},
    'Recovery Stage (years as Trees, 1985–2025)',
    True,
)

# Harvest polygon outlines
for name, geom in harvest_areas.items():
    c = POLY_COLORS[name]
    Map.addLayer(
        ee.Feature(geom, {}),
        {'layerType': 'geeVector', 'strokeColor': c, 'strokeWidth': 3,
         'fillColor': '00000000'},
        name,
        True,
    )

# HJA official boundary perimeter from USFS ArcGIS Feature Service
el.addEsriFeatureService(
    HJA_BOUNDARY_URL,
    viz_params={'strokeColor': 'ffff00', 'strokeWidth': 2.5, 'fillColor': '00000000'},
    name='HJA Boundary',
)

Map.addLayer(
    ee.Feature(study_area, {}),
    {'layerType': 'geeVector', 'strokeColor': 'ffffff', 'strokeWidth': 2,
     'fillColor': '00000000'},
    'H.J. Andrews Study Area',
)

Map.setCenter(-122.20, 44.23, 11)
Map.view()


## 12 · Key Takeaways and Next Steps

### What LCMS tells us about post-harvest recovery at H.J. Andrews

After running this notebook you should be able to answer:

1. **Recovery pace** — How many years does it take for "Trees" land cover to dominate each harvested polygon? Does rate differ by harvest area location (aspect, elevation, proximity to seed source)?

2. **Pre-record recovery** — For WS1 and WS6 (clearcut before 1985), does the Trees% at the *start* of the LCMS record reflect their expected successional age? WS1 at ~20–23 years post-cut should already be in the "young forest" stage; WS6 at ~11 years should still show elevated Shrub%.

3. **Disturbance signal** — Can you find the Tree Removal class in the Change time series for the central and upper basin polygons? How many consecutive years did Tree Removal appear? (Selective harvest may span multiple seasons; clearcut is typically 1–3 years.)

4. **Land use stability** — Does the Sankey show large flows *out* of Forest during the 1985–1995 harvest period, and flows *back* to Forest in subsequent decades after Northwest Forest Plan protections took effect?

5. **Old-growth contrast** — On the Recovery Stage map, what score do the undisturbed old-growth stands receive? Do the older harvest areas (WS1, WS6) "blend in" with old-growth by 2024, or remain detectably different?

---

### Going further

| Idea | How |
|------|-----|
| Use official watershed polygons | Download boundary shapefiles from the [HJ Andrews data portal](https://andrewsforest.oregonstate.edu/data/gis) and replace the `BBox` definitions |
| Add elevation as a recovery predictor | Load `USGS/SRTMGL1_003`, compute slope and aspect, then compare Trees% recovery rate by topographic position |
| Extend to the Willamette NF | Replace `study_area` with `sal.getUSFSForests(forest_name='Willamette')` to see regional context |
| Compare with a drier eastern-Cascades forest | Duplicate the notebook with a Deschutes or Okanogan NF bounding box — recovery is slower in drier climates |
| Export annual land cover maps | Use the EE batch export API (`ee.batch.Export.image.toDrive(...)`) for GeoTIFF outputs |
| Overlay MTBS fire perimeters | Load MTBS from the GEE catalog and check whether any fire intersects the harvest units during the recovery period |

---

### Data citation

> USFS GTAC. (2024). *Landscape Change Monitoring System v2024-10*. USDA Forest Service, Geospatial Technology and Applications Center. [https://www.fs.usda.gov/lcms](https://www.fs.usda.gov/lcms)

> Google Earth Engine catalog: `USFS/GTAC/LCMS/v2024-10`

> H.J. Andrews Experimental Forest LTER Program. Oregon State University, USDA Forest Service PNW Research Station. [https://andrewsforest.oregonstate.edu](https://andrewsforest.oregonstate.edu)